<a href="https://colab.research.google.com/github/MalezhykDaniil/ML/blob/main/%D0%9B%D0%A0_9_%D0%9C%D0%B0%D0%BB%D0%B5%D0%B6%D0%B8%D0%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Малежик Данііл Юрійович
**Група:**
3-10  
**Дата:** 04.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
unique_users = ratings_df['user_id'].nunique()
unique_items = ratings_df['item_id'].nunique()

min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()

missing_values = ratings_df.isnull().sum()

print(f"Кількість унікальних користувачів: {unique_users}")
print(f"Кількість унікальних фільмів: {unique_items}")
print(f"Мінімальна оцінка (rating): {min_rating}")
print(f"Максимальна оцінка (rating): {max_rating}")
print(f"Середня оцінка (rating): {mean_rating:.2f}")
print("Пропущені значення по стовпцях:")
print(missing_values)

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка (rating): 1.0
Максимальна оцінка (rating): 5.0
Середня оцінка (rating): 3.53
Пропущені значення по стовпцях:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

Датасет охоплює оцінки від 943 унікальних користувачів для 1682 унікальних фільмів, що забезпечує достатній обсяг даних для побудови рекомендаційної системи. Оцінки варіюються від 1.0 до 5.0 із середнім значенням 3.53, що вказує на загальну схильність користувачів ставити позитивні або помірно високі бали. Окрім того, у наборі даних повністю відсутні пропущені значення, завдяки чому він не потребує додаткового очищення.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print('Навчальна вибірка:', train_df.shape)
print('Тестова вибірка:', test_df.shape)

Навчальна вибірка: (80000, 4)
Тестова вибірка: (20000, 4)


In [9]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(
    rating_scale=data.reader.rating_scale
)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(
    test_df[['user_id', 'item_id', 'rating']]
    .itertuples(index=False, name=None)
)

print(f"Кількість елементів у train_data: {len(train_data.df)}")
print(f"Кількість елементів у testset: {len(testset)}")

Кількість елементів у train_data: 80000
Кількість елементів у testset: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [10]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [11]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_svd.fit(train_data)

In [12]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_knn.fit(train_data)

In [13]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print('\nНайкращий результат SVD:')
print('RMSE:', grid_search_svd.best_score['rmse'])
print('MAE:', grid_search_svd.best_score['mae'])
print('Параметри за RMSE:', grid_search_svd.best_params['rmse'])

print('\nНайкращий результат KNNBasic:')
print('RMSE:', grid_search_knn.best_score['rmse'])
print('MAE:', grid_search_knn.best_score['mae'])
print('Параметри за RMSE:', grid_search_knn.best_params['rmse'])


Найкращий результат SVD:
RMSE: 0.9435024699944622
MAE: 0.7437495969388217
Параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

Найкращий результат KNNBasic:
RMSE: 1.026697514871081
MAE: 0.8139207318964577
Параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9435 | 0.7438 | 'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05 |
| KNNBasic | 1.0267 | 0.8139 | 'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True} |

Модель SVD є ефективнішою для цього набору даних, оскільки її значення RMSE та MAE нижчі.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [14]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(
    **best_svd_params,
    random_state=42
)

best_svd.fit(trainset)

best_knn = KNNBasic(
    **best_knn_params
)

best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [16]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

print('\nРезультати SVD:')
svd_rmse = accuracy.rmse(
    svd_predictions,
    verbose=True
)
svd_mae = accuracy.mae(
    svd_predictions,
    verbose=True
)

print('\nРезультати KNNBasic:')
knn_rmse = accuracy.rmse(
    knn_predictions,
    verbose=True
)
knn_mae = accuracy.mae(
    knn_predictions,
    verbose=True
)


Результати SVD:
RMSE: 0.9192
MAE:  0.7234

Результати KNNBasic:
RMSE: 1.0120
MAE:  0.7984


In [17]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
}).sort_values('RMSE')

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.919175 | 0.723355 |
| KNNBasic | 1.011953 | 0.798421 |

**Порівняння з CV:** Оцінки на крос-валідації та незалежній тестовій вибірці суттєво не відрізняються.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [19]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
best_model_name = results.iloc[0]['Model']

print(
    f"Найкраща модель за RMSE: {best_model_name}"
)

if results.iloc[0]['Model'] == 'SVD':
    best_model = best_svd
else:
    best_model = best_knn

print(f"Для рекомендацій використовується модель: {results.iloc[0]['Model']}")

try:
    inner_user_id = trainset.to_inner_uid(user_id)

    user_ratings = trainset.ur[inner_user_id]

    print(
        f"Кількість оцінених фільмів користувачем "
        f"{user_id}: {len(user_ratings)}"
    )

    all_inner_items = set(trainset.all_items())

    rated_inner_items = {
        inner_item_id
        for inner_item_id, rating in user_ratings
    }

    unrated_inner_items = all_inner_items - rated_inner_items

    recommendations = []

    for inner_item_id in unrated_inner_items:
        raw_item_id = trainset.to_raw_iid(inner_item_id)

        predicted_rating = best_model.predict(
            uid=user_id,
            iid=raw_item_id
        ).est

        recommendations.append({
            'item_id': raw_item_id,
            'predicted_rating': predicted_rating
        })

    recommendations_df = (
        pd.DataFrame(recommendations)
        .sort_values(
            by='predicted_rating',
            ascending=False
        )
        .head(10)
        .reset_index(drop=True)
    )

    print(
        f"\nТоп-10 рекомендацій для користувача {user_id}:"
    )

    display(recommendations_df)

except ValueError:
    print(
        f"Користувача з ID {user_id} "
        "немає в навчальній вибірці."
    )

Найкраща модель за RMSE: SVD
Для рекомендацій використовується модель: SVD
Кількість оцінених фільмів користувачем 196: 29

Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [✓] Дані `ml-100k` завантажено та перевірено.
- [✓] Є розбиття 80/20 з фіксованим `random_state`.
- [✓] `Reader`, `train_data` і `testset` створені коректно.
- [✓] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [✓] Для SVD і KNNBasic показано параметри та результати CV.
- [✓] Моделі навчено на повній навчальній частині.
- [✓] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [✓] Наведено порівняльну таблицю та інтерпретацію.
- [✓] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [✓] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:**

У ході лабораторної роботи було розроблено та порівняно дві моделі рекомендаційних систем: SVD та KNNBasic. За результатами кросвалідації кращою виявилася модель SVD з RMSE = 0.9435 та MAE = 0.7438. Для неї було підібрано такі гіперпараметри: 50 прихованих факторів, швидкість навчання 0.01 та регуляризація 0.05. Для KNNBasic найкращими параметрами стали 40 сусідів, мінімальна кількість сусідів 1 та косинусна метрика схожості. На тестовій вибірці SVD отримала RMSE = 0.919175 та MAE = 0.723355, тоді як KNNBasic — RMSE = 1.011953 та MAE = 0.798421. Результати тестування не мають суттєвих відмінностей від результатів кросвалідації, що свідчить про стабільність моделей. На основі SVD було сформовано топ-10 рекомендацій для користувача, виключивши фільми, які він уже оцінював. Основними обмеженнями системи є недостатність інформації про нових користувачів і фільми, а також залежність точності рекомендацій від наявних оцінок.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.